# Laboratorio 04 â€” Pipeline Silver Metadata-Driven y Databricks Jobs

**Semana:** 04 | **Actividad de referencia:** Actividad 04  
**Modalidad:** Individual | **Entorno:** Databricks (Unity Catalog)

---

## Instrucciones generales

DiseÃ±a un archivo `silver_config.yml` para tu dataset, construye el notebook Silver que aplica transformaciones declarativas a partir de esa config, y describe cÃ³mo lo organizarÃ­as como un Databricks Job de mÃºltiples tareas (Bronze â†’ Silver â†’ Gold).

## Parte 1 â€” DescripciÃ³n del dataset y diseÃ±o de la capa Silver

1. **Nombre y fuente** del dataset ya ingestado en Bronze (lab anterior).
2. **Transformaciones Silver:** Â¿QuÃ© limpiezas y estandarizaciones aplicarÃ¡s? (renombrar columnas, castear tipos, filtrar nulos, normalizar strings, calcular campos derivados, etc.)
3. **Estrategia de escritura:** Â¿UsarÃ¡s `overwrite`, `append` o `MERGE INTO`? Â¿Por quÃ©?
4. **Estructura del Job:** Describe las tareas que tendrÃ­a el Databricks Job para ejecutar Bronze â†’ Silver â†’ Gold en orden.

**Escribe tu respuesta aquÃ­:**

## Parte 2 â€” Crear el archivo silver_config.yml

In [ ]:
import os, yaml

CONFIG_DIR   = "/tmp/lab04_04"
CONFIG_PATH  = f"{CONFIG_DIR}/silver_config.yml"
os.makedirs(CONFIG_DIR, exist_ok=True)

# Ajusta todos los valores a tu dataset real
silver_yaml = """
entorno: dev

dev:
  tablas:
    - nombre: mi_dataset
      origen: workspace.default.bronze_dev_mi_dataset
      destino: workspace.default.silver_dev_mi_dataset
      modo_escritura: overwrite

      filtros:
        - "columna_clave IS NOT NULL"
        - "columna_numerica > 0"

      renombrar:
        columna_vieja_1: columna_nueva_1
        columna_vieja_2: columna_nueva_2

      castear:
        columna_fecha: timestamp
        columna_id:    integer

      normalizar_strings:
        - columna_texto_1
        - columna_texto_2

      columnas_derivadas:
        - nombre: anio
          expresion: "YEAR(columna_fecha)"
        - nombre: mes
          expresion: "MONTH(columna_fecha)"

      merge_key: columna_clave
"""

with open(CONFIG_PATH, "w") as f:
    f.write(silver_yaml)

print(f"âœ“ silver_config.yml guardado en {CONFIG_PATH}")

In [ ]:
# Cargar y validar la config
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

entorno  = config["entorno"]
env_cfg  = config[entorno]

print(f"Entorno: {entorno}")
for t in env_cfg["tablas"]:
    print(f"  {t['nombre']}: {t['origen']} â†’ {t['destino']} (modo={t['modo_escritura']})")

## Parte 3 â€” Perfil tÃ©cnico de la tabla Bronze de origen

In [ ]:
from pyspark.sql import functions as F

tabla_cfg = env_cfg["tablas"][0]
df_bronze = spark.table(tabla_cfg["origen"])

print(f"Bronze: {df_bronze.count():,} filas | {len(df_bronze.columns)} columnas")
df_bronze.printSchema()

In [ ]:
# EstadÃ­sticas rÃ¡pidas antes de transformar
df_bronze.describe().show(truncate=False)

## Parte 4 â€” Aplicar transformaciones declarativas desde la config

In [ ]:
def aplicar_transformaciones_silver(df, cfg):
    """Aplica las transformaciones declaradas en silver_config a un DataFrame."""

    # 1. Filtros
    for filtro in cfg.get("filtros", []):
        df = df.filter(filtro)
        print(f"  Filtro aplicado: {filtro} â†’ {df.count():,} filas")

    # 2. Renombrar columnas
    for viejo, nuevo in cfg.get("renombrar", {}).items():
        if viejo in df.columns:
            df = df.withColumnRenamed(viejo, nuevo)
            print(f"  Renombrado: {viejo} â†’ {nuevo}")
        else:
            print(f"  âš  Columna '{viejo}' no encontrada para renombrar")

    # 3. Castear tipos
    for col_nombre, tipo in cfg.get("castear", {}).items():
        if col_nombre in df.columns:
            df = df.withColumn(col_nombre, F.col(col_nombre).cast(tipo))
            print(f"  Cast: {col_nombre} â†’ {tipo}")

    # 4. Normalizar strings (trim + lower)
    for col_nombre in cfg.get("normalizar_strings", []):
        if col_nombre in df.columns:
            df = df.withColumn(col_nombre, F.trim(F.lower(F.col(col_nombre))))

    # 5. Columnas derivadas con expresiones SQL
    for col_derivada in cfg.get("columnas_derivadas", []):
        df = df.withColumn(col_derivada["nombre"], F.expr(col_derivada["expresion"]))
        print(f"  Derivada: {col_derivada['nombre']} = {col_derivada['expresion']}")

    # 6. Agregar timestamp Silver
    df = df.withColumn("_silver_ts", F.current_timestamp())

    return df

df_silver = aplicar_transformaciones_silver(df_bronze, tabla_cfg)
print(f"\nâœ“ Silver listo: {df_silver.count():,} filas | {len(df_silver.columns)} columnas")
df_silver.printSchema()

In [ ]:
# Muestra primeras filas del DataFrame Silver
df_silver.show(5, truncate=False)

**AnÃ¡lisis de transformaciones:** Â¿QuÃ© filas se perdieron en los filtros? Â¿Hubo columnas que no existÃ­an para renombrar? Â¿Los tipos casteados son correctos?

## Parte 5 â€” Escribir la capa Silver (overwrite o MERGE INTO)

In [ ]:
destino = tabla_cfg["destino"]
modo    = tabla_cfg["modo_escritura"]

if modo == "merge":
    # Estrategia MERGE INTO â€” idempotente
    merge_key = tabla_cfg["merge_key"]
    df_silver.createOrReplaceTempView("silver_updates")
    spark.sql(f"""
        MERGE INTO {destino} AS target
        USING silver_updates AS src
        ON target.{merge_key} = src.{merge_key}
        WHEN MATCHED THEN
            UPDATE SET *
        WHEN NOT MATCHED THEN
            INSERT *
    """)
    print(f"âœ“ MERGE INTO {destino} completado")
else:
    df_silver.write.format("delta").mode(modo).saveAsTable(destino)
    print(f"âœ“ Escrito en {destino} (modo={modo})")

print(f"  Filas en Silver: {spark.table(destino).count():,}")

In [ ]:
# Historial de versiones Delta de la tabla Silver
spark.sql(f"DESCRIBE HISTORY {destino}").select("version", "timestamp", "operation").show(5)

## Parte 6 â€” Preguntas de negocio en la capa Silver

In [ ]:
# Pregunta 1 â€” usando la tabla Silver
spark.sql(f"""
    SELECT -- tu consulta de negocio
    FROM {destino}
    LIMIT 15
""").show(truncate=False)

**ConclusiÃ³n pregunta 1:**

In [ ]:
# Pregunta 2 â€” compara Bronze vs Silver para verificar la calidad de la transformaciÃ³n
print("Bronze â†’ Silver comparaciÃ³n de conteos:")
print(f"  Bronze: {spark.table(tabla_cfg['origen']).count():,}")
print(f"  Silver: {spark.table(destino).count():,}")
pct_pÃ©rdida = round((1 - spark.table(destino).count() / spark.table(tabla_cfg["origen"]).count()) * 100, 1)
print(f"  PÃ©rdida por filtros: {pct_pÃ©rdida}%")

**ConclusiÃ³n pregunta 2:** Â¿La pÃ©rdida de registros es aceptable para tu negocio?

## Parte 7 â€” Arquitectura del Databricks Job (teÃ³rico)

Dibuja o describe en markdown la estructura del Job multi-tarea que orquestarÃ­a tu pipeline completo:

```
Tarea 01: Bronze Ingestion
  notebook: /semana_04/laboratorios/lab_02_bronze_generico
  parÃ¡metros: { "tabla_destino": "workspace.default.bronze_dev_mi_dataset" }
  dependencias: ninguna

Tarea 02: Silver Transformation
  notebook: /semana_04/laboratorios/lab_04_silver_jobs
  parÃ¡metros: { "entorno": "dev" }
  dependencias: [Tarea 01]

Tarea 03: Gold Aggregation
  notebook: (crea el esqueleto abajo)
  parÃ¡metros: { "tabla_silver": "workspace.default.silver_dev_mi_dataset" }
  dependencias: [Tarea 02]
```

**Edita la arquitectura arriba con tus nombres reales. Luego responde:**
1. Â¿Por quÃ© es importante el orden de dependencias?
2. Â¿QuÃ© pasarÃ­a si la Tarea 01 falla? Â¿La Tarea 02 deberÃ­a ejecutarse?
3. Â¿CÃ³mo notificarÃ­as al equipo si alguna tarea falla?

In [ ]:
# Esqueleto de la capa Gold (agrega como te convenga)
tabla_silver = destino
tabla_gold   = destino.replace("silver", "gold")

df_gold = spark.table(tabla_silver) \
    .groupBy("columna_particion") \
    .agg(
        F.count("*").alias("total"),
        F.avg("columna_numerica").alias("promedio")
    ) \
    .withColumn("_gold_ts", F.current_timestamp())

# Comenta la siguiente lÃ­nea si no quieres crear la tabla Gold todavÃ­a
# df_gold.write.format("delta").mode("overwrite").saveAsTable(tabla_gold)

df_gold.show(15, truncate=False)
print(f"âœ“ Gold preview: {df_gold.count()} grupos")

## Parte 8 â€” ReflexiÃ³n final

1. Â¿QuÃ© ventaja tiene definir las transformaciones en YAML en lugar de hard-codearlas en el notebook?
2. Â¿En quÃ© caso usarÃ­as `MERGE INTO` sobre `overwrite`? Â¿CuÃ¡l tiene mejor rendimiento?
3. Â¿CÃ³mo protegerÃ­as las tablas Silver y Gold de escrituras accidentales en el entorno de producciÃ³n?
4. Â¿QuÃ© agregarÃ­as al `silver_config.yml` para soportar validaciones de calidad mÃ¡s complejas (p.ej., rango de valores, expresiones regulares)?

---

## Entrega en Git

```bash
# Copia el template a tu carpeta (solo la primera vez)
# cp semana_04/laboratorios/lab_04_silver_jobs.ipynb semana_04/laboratorios/<tu-nombre>/lab_04_silver_jobs.ipynb

git add semana_04/laboratorios/<tu-nombre>/lab_04_silver_jobs.ipynb
git commit -m "lab: semana04 lab04 silver metadata MERGE Jobs <nombre-dataset> - <tu-nombre>"
git push origin develop
```